<a href="https://colab.research.google.com/github/wallynovak/fumarase_kinetics/blob/main/01%20-%20Tecan_kinetic_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Tecan Kinetic Processing

This notebook takes an unedited csv file and processes the kinetics in a stepwise manner. Please upload your csv file before proceeding.


## Step 1. Enter your filename

In [ ]:
#@title # Simply click the run button. It will ask you to enter your filename.
#@markdown <br> Do not edit this cell, just run it.
fname = input("Please enter the name of your CSV file (e.g. \"Wally Wabash Kinetics.csv\"): ")
file_path = "/content/"+fname

print(f"File path: {file_path}")

## Step 2. Importing the kinetic data

Here we identify the data to import into a python dataframe using the keywords "kinetic measurement" and "End Time". In the next section we will clean up the dataframe.

In [ ]:
import pandas as pd
import io

try:
    # Read all lines from the CSV file to efficiently find start and end markers
    with open(file_path, 'r') as f:
        all_lines = f.readlines()

    start_row_index = -1  # 0-indexed line number of the header row for the DataFrame
    end_row_index = -1    # 0-indexed line number *after* the last data row

    found_start_marker = False
    for i, line in enumerate(all_lines):
        # Assuming the CSV uses comma as a delimiter. Adjust line.split(',') if needed.
        first_cell_value = line.split(',')[0].strip()

        if not found_start_marker:
            if first_cell_value == "kinetic measurement":
                start_row_index = i
                found_start_marker = True
        elif found_start_marker:
            # Once 'kinetic measurement' is found, look for 'End Time'
            if first_cell_value == "End Time":
                end_row_index = i-2 # back it up if we read 'End Time'
                break # Stop searching once 'End Time' is found

    if start_row_index == -1:
        print(f"Error: 'kinetic measurement' not found in the first column of '{file_path}'.")
        df = pd.DataFrame() # Initialize an empty DataFrame
    else:
        # Calculate the number of rows to read
        if end_row_index == -1:
            # 'End Time' not found, read until the end of the file from the start_row_index
            nrows_to_read = len(all_lines) - start_row_index
        else:
            # Read from start_row_index up to, but not including, end_row_index
            nrows_to_read = end_row_index - start_row_index

        # Import the relevant section of the CSV into a Pandas DataFrame
        # skiprows: Skips lines from the beginning of the file up to the start_row_index.
        # header=0: The first line *after* skipping (i.e., the line at start_row_index) is used as the header.
        # nrows: Reads the specified number of rows after the header.
        df = pd.read_csv(
            file_path,
            skiprows=start_row_index,
            nrows=nrows_to_read,
            header=0 # The line at `start_row_index` is used as the header
        )
        print("DataFrame 'df' created successfully with the kinetic measurement data:")
        print(df.head())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the path is correct and the file exists.")
    df = pd.DataFrame() # Initialize an empty DataFrame on error
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = pd.DataFrame() # Initialize an empty DataFrame on error

## Step 3. Clean the dataframe.

Notice the data has columns we don't need and lots of cells that are blank (NaN). The code below will retain only the data needed.

In [ ]:
# Promote the first row to be the column headers and then drop the original first row
if df.columns[0] != 'Time [min]': # makes sure this wasn't already done
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)

    # Clean up column names by stripping whitespace
    df.columns = df.columns.str.strip()

    # Define columns to remove explicitly
    columns_to_remove_explicit = ['Cycle Nr.', 'Temp. [°C]']

    # Identify columns that exist in the DataFrame from the explicit list
    existing_explicit_cols = [col for col in columns_to_remove_explicit if col in df.columns]

    # Identify columns that contain any NaN values
    # Note: This checks for *any* NaN. If you only want to drop columns that are *entirely* NaN,
    # you would use `df.isnull().all()` instead of `df.isnull().any()`.
    cols_with_nan = df.columns[df.isnull().any()].tolist()

    # Combine all columns to drop and remove duplicates
    all_cols_to_drop = list(set(existing_explicit_cols + cols_with_nan))

    # Drop the identified columns
    df = df.drop(columns=all_cols_to_drop)
else:
    print("Column headers are already correct")

if 'Time [s]' in df.columns:
    df['Time [s]'] = df['Time [s]'].astype(float) / 60
    df = df.rename(columns={'Time [s]': 'Time [min]'})
    print("Time [s] successfully converted to Time [min].")
else:
    print("'Time [s]' column not found. It might have already been converted to 'Time [min]' or renamed.")

print("DataFrame cleaned successfully. Here's the head of the updated DataFrame:")
print(df.head())

## Step 4. Output the dataframe to a csv file
The data can be printed and included in your lab notebook or used in excel.

In [ ]:
output_filename = 'processed_data.csv'
df.to_csv(output_filename, index=False)
print(f"DataFrame successfully saved to {output_filename}")
df

## Step 5. Plot the data

Next lets plot each of our data runs to ensure it looks okay. You should take a screenshot of these graphs and print them for your lab notebook. Clearly label these graphs in your notebook.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Dynamically determine groups and column_suffixes from df columns
# Exclude 'Time [min]' column
data_columns = [col for col in df.columns if col != 'Time [min]']

groups = sorted(list(set([re.match(r'([A-Z]+)', col).group(1) for col in data_columns if re.match(r'([A-Z]+)', col)])))
# Assuming suffixes are numeric and follow the group prefix
column_suffixes = sorted(list(set([re.sub(r'[A-Z]+', '', col) for col in data_columns if re.sub(r'[A-Z]+', '', col)])))

print(f"Dynamically determined groups: {groups}")
print(f"Dynamically determined column suffixes: {column_suffixes}")

# Create a figure and a set of subplots (adjust rows/cols based on number of groups)
rows = (len(groups) + 1) // 2 # 2 columns, so calculate rows needed
if rows == 0: rows = 1 # Ensure at least 1 row for single group case
fig, axes = plt.subplots(nrows=rows, ncols=2, figsize=(15, 5 * rows), sharex=True, sharey=True)
axes = axes.flatten() # Flatten the array of axes for easy iteration

# Hide unused subplots if any
for i in range(len(groups), len(axes)):
    fig.delaxes(axes[i])

# Iterate through each group to create a subplot
for i, group_prefix in enumerate(groups):
    ax = axes[i]
    y_cols = [f'{group_prefix}{suffix}' for suffix in column_suffixes]

    # Check if the 'Time [min]' column exists and convert to float
    if 'Time [min]' not in df.columns:
        print(f"Error: 'Time [min]' column not found in the DataFrame. Cannot plot Group {group_prefix}.")
        ax.set_title(f'Group {group_prefix} (Time [min] Missing)')
        ax.set_xlabel('Time [min]')
        ax.set_ylabel('deltaAU')
        continue

    time_data = df['Time [min]'].astype(float)

    # Plot each series for the current group
    for col in y_cols:
        if col in df.columns:
            sns.lineplot(x=time_data, y=df[col].astype(float), ax=ax, label=col)
        else:
            print(f"Warning: Column '{col}' not found for Group {group_prefix}.")

    ax.set_title(f'Group {group_prefix} Kinetics')
    ax.set_xlabel('Time [min]')
    ax.set_ylabel('deltaAU')
    ax.legend(title='Well', loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

## Step 6. Determine the Slopes of the graphs in deltaAU/min

Here we will calculate all of the slopes, the R^2 values and the average slopes for each group +/- the standard deviation.

In [ ]:
from scipy.stats import linregress
import numpy as np
import re

# Dynamically determine groups and column_suffixes from df columns
# Exclude 'Time [min]' column
data_columns = [col for col in df.columns if col != 'Time [min]']

groups = sorted(list(set([re.match(r'([A-Z]+)', col).group(1) for col in data_columns if re.match(r'([A-Z]+)', col)])))
# Assuming suffixes are numeric and follow the group prefix
column_suffixes = sorted(list(set([re.sub(r'[A-Z]+', '', col) for col in data_columns if re.sub(r'[A-Z]+', '', col)])))

print(f"Dynamically determined groups for analysis: {groups}")
print(f"Dynamically determined column suffixes for analysis: {column_suffixes}")

print("--- Linear Regression Analysis ---")
print("Each slope is in deltaAU/min.")
print("----------------------------------")

all_group_slopes = {}

for group_prefix in groups:
    group_slopes = []
    print(f"\nGroup {group_prefix}:")

    # Check if 'Time [min]' column exists
    if 'Time [min]' not in df.columns:
        print(f"  Error: 'Time [min]' column not found. Cannot calculate slopes for Group {group_prefix}.")
        continue

    time_data = df['Time [min]'].astype(float)

    for suffix in column_suffixes:
        col = f'{group_prefix}{suffix}'
        if col in df.columns:
            y_data = df[col].astype(float)

            # Perform linear regression
            # Only consider non-NaN values for regression if any
            valid_indices = ~y_data.isnull()
            if valid_indices.any():
                slope, intercept, r_value, p_value, std_err = linregress(time_data[valid_indices], y_data[valid_indices])
                r_squared = r_value**2
                group_slopes.append(slope)
                print(f"  Column {col}: Slope = {slope:.4f}, R² = {r_squared:.4f}")
            else:
                print(f"  Column {col}: No valid data points for regression.")
        else:
            print(f"  Column {col}: Not found in DataFrame.")

    if group_slopes:
        avg_slope = np.mean(group_slopes)
        std_dev_slope = np.std(group_slopes)
        all_group_slopes[group_prefix] = {'average_slope': avg_slope, 'std_dev_slope': std_dev_slope}
        print(f"  Average Slope for Group {group_prefix}: {avg_slope:.4f} +/- {std_dev_slope:.4f}")
    else:
        print(f"  No valid slopes calculated for Group {group_prefix}.")

print("\n--- Summary of Group Slopes ---")
for group, data in all_group_slopes.items():
    print(f"Group {group}: Average Slope = {data['average_slope']:.4f} +/- {data['std_dev_slope']:.4f}")

## Step 7. Correct your slopes to the pathlength

The above slopes are not corrected for the pathlength. Calculate the average pathlength from ONLY YOUR BLANK wells (25 mM malate, no enzyme). Why? Notice that high activity wells have altered pathlengths, suggesting the product of the reaction, fumarate, interferes with the pathlength reading. Run the cell below, inputting your pathlength to correct your data to a 1 cm pathlength.

In [ ]:
# Prompt the user to input the average pathlength
pathlength_str = input("Please enter the average pathlength in cm (e.g., 0.5): ")
try:
    pathlength = float(pathlength_str)
except ValueError:
    print("Invalid input. Please enter a numeric value for pathlength.")
    # Optionally, handle this error more robustly, e.g., by exiting or asking again
    pathlength = 1.0 # Default to 1 cm to avoid further errors, or handle as appropriate

print(f"\nCorrecting slopes to a 1 cm pathlength using an average pathlength of {pathlength:.2f} cm:\n")

corrected_group_slopes = {}
for group, data in all_group_slopes.items():
    original_avg_slope = data['average_slope']
    original_std_dev_slope = data['std_dev_slope']

    # Correct by dividing by pathlength (to normalize to 1 cm, since all measurements were at 'pathlength')
    corrected_avg_slope = original_avg_slope / pathlength
    corrected_std_dev_slope = original_std_dev_slope / pathlength

    corrected_group_slopes[group] = {
        'average_slope': corrected_avg_slope,
        'std_dev_slope': corrected_std_dev_slope
    }
    print(f"Group {group}: Corrected Average Slope = {corrected_avg_slope:.4f} +/- {corrected_std_dev_slope:.4f} deltaAU/min (Original: {original_avg_slope:.4f} +/- {original_std_dev_slope:.4f})")


## Step 8. Copy this data into your lab notebook and proceed to the next notebook (Fumarase_kinetics.ipynb) to calculate activity in U/mg and determine the kinetic constants.